# Why do we program with classes?

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*

This notebook is about **why**, not about how.

The syntax &mdash; `class`, `self`, constructors, inheritance &mdash; is covered in
[classes_intro](./classes_intro.ipynb), [classes](./classes.ipynb) and
[classes_inheritance](./classes_inheritance.ipynb). Here we only ask what problem
classes solve, and when they are the wrong answer.

## 1. The question

You already know how to write functions. Functions are enough to compute anything.

So why does almost every serious scientific code &mdash; ROOT, Geant4, astropy, scikit-learn,
even `numpy` itself &mdash; hand you **objects** instead of functions?

The answer is not "because it is modern", and it is not "because it is elegant".
It is that three specific things become painful as a program grows, and classes make
them go away.

## 2. Let us start from a problem

We want to simulate particles moving under a force. With functions alone:

In [ ]:
def step(x, v, m, force, dt):
    a = force / m
    return x + v * dt, v + a * dt

x, v, m = 0.0, 1.0, 2.0
for _ in range(3):
    x, v = step(x, v, m, force=-1.0, dt=0.1)
print(f"x = {x:.3f}, v = {v:.3f}")

This works. It is clear. For one particle and one afternoon, stop here &mdash;
you do not need a class.

The trouble starts when the program grows.

## 3. Symptom 1: data that belongs together travels apart

`x`, `v` and `m` describe **one** particle, but the language does not know that.
Nothing stops us from mixing them up.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — A second particle</b></p>

The next cell moves a second particle. It runs without errors. Is the result right?

</div>

In [ ]:
# a second particle
x2, v2, m2 = 5.0, -1.0, 3.0

x2, v2 = step(x2, v, m2, force=-1.0, dt=0.1)
print(f"x2 = {x2:.3f}, v2 = {v2:.3f}")

Look at the call: `v`, not `v2`. No error, no warning, a wrong physics result. With three particles and three
quantities each you are carrying nine loose variables, and the chance of one of
these mistakes approaches certainty.

**A class binds the data that belongs together into one thing you cannot take apart by accident.**

In [ ]:
class Particle:
    def __init__(self, x, v, m):
        self.x, self.v, self.m = x, v, m

    def step(self, force, dt):
        self.v += (force / self.m) * dt
        self.x += self.v * dt

p = Particle(5.0, -1.0, 3.0)
p.step(force=-1.0, dt=0.1)
print(f"x = {p.x:.3f}, v = {p.v:.3f}")

Now `step` cannot be called with *this* particle's position and *that* particle's
velocity.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Same physics?</b></p>

Compare the function `step` (first cell) and the method `Particle.step`. For the same
particle, `x = 5, v = -1, m = 3`, one step with `force = -1` and `dt = 0.1`: do they
give the same `x`? Run the next cell. Which one is right? What happened?

</div>

In [ ]:
print(step(5.0, -1.0, 3.0, -1.0, 0.1))
q = Particle(5.0, -1.0, 3.0)
q.step(-1.0, 0.1)
print(q.x, q.v)

<details>
<summary><b>Solution</b></summary>

Both are valid, but they are **different algorithms**. The function updates the position
with the *old* velocity (explicit Euler); the method updates the velocity first and moves
with the *new* one (semi-implicit, or symplectic, Euler). The second one is better for
oscillators: its energy error stays bounded.

The lesson is not about Euler: rewriting code into a class (a *refactor*) changed the
results, and nobody noticed. A refactor must keep the tests passing, so write the tests
**before** you refactor. (To make them agree, move `self.x += self.v * dt` before the
velocity update.)

</details>

## 4. Symptom 2: things that must always be true

A mass is positive. A direction vector has unit length. A probability lies in [0, 1].
These are **invariants**: statements that must hold at every moment.

With loose variables, every piece of code that touches the data must remember to
check. With a class, there is **one place** where it is checked.

In [ ]:
class Particle:
    def __init__(self, x, v, m):
        if m <= 0:
            raise ValueError(f"mass must be positive, got {m}")
        self.x, self.v, self.m = x, v, m

    def step(self, force, dt):
        self.v += (force / self.m) * dt
        self.x += self.v * dt

try:
    Particle(0.0, 1.0, m=-2.0)
except ValueError as e:
    print("caught:", e)

The object cannot be **created** in an invalid state. That is worth more than it sounds:
most long debugging sessions are spent finding out *where* the data went wrong,
not *what* went wrong.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Broken later?</b></p>

The constructor refuses a negative mass. **Decide**: does this raise an error?

```python
p = Particle(0.0, 1.0, m=2.0)
p.m = -2.0
```

</div>

In [ ]:
p = Particle(0.0, 1.0, m=2.0)
p.m = -2.0
print(p.m)

<details>
<summary><b>Solution</b></summary>

No: the check runs only in `__init__`, and `p.m = -2.0` is a plain assignment. Keeping
an invariant **after** creation needs a *property*, a method that runs when the attribute
is assigned: see *Keeping invariants: properties* in [Writing classes](./classes.ipynb).

</details>

## 5. Symptom 3: the same operation on different things

This is the one that actually matters.

Suppose the force can come from several models. Written procedurally, the simulation
loop has to know about every one of them:

In [ ]:
def force_of(kind, x, v):
    if kind == "spring":
        return -1.0 * x
    elif kind == "gravity":
        return -9.8
    elif kind == "drag":
        return -0.5 * v
    else:
        raise ValueError(kind)

print([round(force_of(k, 2.0, 1.0), 2) for k in ("spring", "gravity", "drag")])

Every new model means editing this function.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — How many places?</b></p>

A fourth force model arrives: a constant push. In a real code the `if/elif` chain of
`force_of` appears in the simulation, in the plotting and in the output. **Decide**: how
many places must you edit in this version? And how many existing lines will change in
the version with objects, that comes next?

</div>

### The same thing with objects

Each model is an object that knows how to compute its own force.

In [ ]:
class Spring:
    def __init__(self, k):    self.k = k
    def force(self, x, v):    return -self.k * x

class Gravity:
    def __init__(self, g, m): self.g, self.m = g, m   # m: the particle's mass
    def force(self, x, v):    return -self.m * self.g

class Drag:
    def __init__(self, b):    self.b = b
    def force(self, x, v):    return -self.b * v

for model in (Spring(1.0), Gravity(9.8, 1.0), Drag(0.5)):
    print(f"{type(model).__name__:8s} -> {model.force(2.0, 1.0):6.2f}")

### And now the point

The code that *uses* the models never mentions any of them:

In [ ]:
def simulate(particle, model, dt, n):
    for _ in range(n):
        particle.step(model.force(particle.x, particle.v), dt)
    return particle.x

print(f"spring : {simulate(Particle(1.0, 0.0, 1.0), Spring(1.0),  0.01, 100):.4f}")
print(f"drag   : {simulate(Particle(0.0, 1.0, 1.0), Drag(0.5),    0.01, 100):.4f}")

Adding a fourth force model requires **no change at all** to `simulate`.
It only has to have a method called `force`.

> This is the real argument for classes: **the code that uses a thing does not
> need to know what the thing is** &mdash; only what it can do.

That is what people mean by *polymorphism*, and it is why every simulation framework
you will meet is built this way.

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Is a function enough?</b></p>

`simulate(p, lambda x, v: -x, 0.01, 100)` would work too, if `simulate` called the
function directly. Then why classes for the force models? Think of a model with several
parameters, one that keeps a state (a random generator, a table read from a file), or one
that must offer several methods (`force`, `potential`, `describe`).

</div>

<details>
<summary><b>Solution</b></summary>

For a single, stateless formula, a function (or a `functools.partial`) is enough, and simpler.
A class wins when the model carries **parameters and state** that belong together, or
when the code needs **several** operations from the same model: `force` and `potential`
must use the same `k`. Functional and object-oriented code are not rivals: the last
chapter, [Functional or object-oriented?](./fp_or_oop.ipynb), compares them on one
problem.

</details>

## 6. You have been using classes all along

You do not get to opt out. In Python, everything is an object.

In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
print(type(a))
print(a.mean(), a.sum(), a.shape)     # methods and attributes of an object

s = "hello"
print(type(s), s.upper())

`a.mean()` is a method call on an instance of `numpy.ndarray`. `s.upper()` is a
method call on an instance of `str`.

So the question is never *"should I use objects?"* &mdash; you already do.
It is only *"should I write a new kind of object myself?"*.

## 7. So what is a class, really?

Three things bound together:

| | |
|---|---|
| **State** | the data that describes one instance (`x`, `v`, `m`) |
| **Behaviour** | what you can do to it (`step`, `force`) |
| **A promise** | the *interface*: any object with a `force(x, v)` method can be used by `simulate` |

The promise is the part people forget, and the part that does the work.

## 8. The four words people will throw at you

| Word | What it actually means | Honest weight |
|---|---|---|
| **Encapsulation** | keep the data and the code that maintains it together | useful |
| **Abstraction** | expose what it does, hide how | useful |
| **Polymorphism** | different objects, same interface | **the reason classes exist** |
| **Inheritance** | a class built on another | overused; see below |

Textbooks give all four equal billing. They are not equal.

### A warning about inheritance

Inheritance is the most abused idea in object-oriented programming. A deep hierarchy
(`Detector` &rarr; `GaseousDetector` &rarr; `TPC` &rarr; `HighPressureTPC` &rarr; ...) is
very hard to change later, because every class depends on all its ancestors.

A common rule of thumb in the Python community:

> **Prefer composition to inheritance.** An object that *has* a `Geometry` and a
> `ReadOut` is easier to change than one that *is* a subclass of five things.

Use inheritance when you genuinely mean "is a kind of", and keep it one level deep
if you can.

## 9. When *not* to write a class

Python is not Java. There is no rule that says code must live in a class,
and a class that is not carrying state is just a function wearing a costume.

In [ ]:
# An anti-pattern: no state, one method
class EnergyCalculator:
    def compute(self, m):
        return m * 299792458**2

print(EnergyCalculator().compute(1.0))

# The same thing, better
def energy(m):
    return m * 299792458**2

print(energy(1.0))

**If it has no state and one method, it is a function. Write the function.**

### Lighter alternatives

Often you want to group data, but not behaviour. Python has cheaper tools than a full class:

In [ ]:
from dataclasses import dataclass

@dataclass
class Measurement:
    value: float
    error: float
    units: str = "MeV"

m = Measurement(125.3, 0.4)
print(m)                       # a readable repr, for free
print(m.value + m.error)

| Tool | Use it when |
|---|---|
| a plain **function** | no state to keep |
| a **tuple** or **dict** | throwaway grouping, used in one place |
| `@dataclass` / `NamedTuple` | data with names, little or no behaviour |
| a **class** | state *and* behaviour, invariants, or several implementations of one interface |
| a **numpy array** | many identical things &mdash; see below |

## 10. The cost, in physics

A class per particle is fine for ten particles. For ten million, think about what
changes.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — How much slower?</b></p>

Half a million particles, one step. **Bet** before running: is a list of objects about
as fast as numpy arrays, 10 times slower, or 100 times slower?

</div>

In [ ]:
import numpy as np, time

N = 500_000

class P:
    def __init__(self, x, v): self.x, self.v = x, v

ps = [P(0.0, 1.0) for _ in range(N)]
t0 = time.perf_counter()
for p in ps:
    p.x += p.v * 0.01
t_obj = time.perf_counter() - t0

x = np.zeros(N); v = np.ones(N)
t0 = time.perf_counter()
x += v * 0.01
t_arr = time.perf_counter() - t0

print(f"one object per particle : {t_obj*1e3:7.1f} ms")
print(f"numpy arrays            : {t_arr*1e3:7.1f} ms")
print(f"speed-up                : {t_obj/t_arr:7.0f}x")

Each object carries its own memory overhead, and the loop runs in Python instead of
in C. The numbers depend on your machine.

So the right design for a physics code is usually **not** a million small objects.
It is a *few* objects, each holding **arrays**: one `Event` with arrays of hits,
one `Detector` with arrays of channels.

Objects for the structure; arrays for the numbers.

## 11. When to reach for a class: a checklist

Write a class when you can answer *yes* to at least one of these:

1. Do several pieces of data always travel together? (With no behaviour: a `dataclass`.)
2. Is there something that must always be true about them?
3. Will there be **more than one implementation** of the same idea,
   and do you want the rest of the code not to care which?
4. Does the thing have a lifetime &mdash; created, used many times, changed, closed?

If the answer to all four is *no*, write a function.

## 12. Practice

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Class, function or dataclass?</b></p>

The code below is procedural. **Decide** what should become a class, what should stay a
function, and what should be a `dataclass`. Justify each choice with the checklist
above; do not write the code yet.

```python
def read_spectrum(path): ...
def calibrate(channels, slope, offset): ...
def subtract_background(counts, bkg): ...
def fit_peak(energies, counts, e_min, e_max): ...
def plot(energies, counts, title): ...
```

</div>

<details>
<summary><b>Solution</b></summary>

One reasonable answer (there are others):

* a `Spectrum` **class** holding `channels`, `counts` and, after calibration, `energies`
  (data that travels together,
  item 1; counts never negative, item 2), with methods `calibrate` (returns a new
  `Spectrum` with `energies = slope * channels + offset`), `subtract_background`,
  `fit_peak` and `plot`;
* the calibration `(slope, offset)` and the fit result `(mean, sigma, area, errors)` as
  **dataclasses**: data with names and no behaviour;
* `read_spectrum(path)` stays a **function** (or a `Spectrum.from_file` classmethod) that
  returns a `Spectrum`.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A fourth force</b></p>

Write a fourth force model, a constant push `force(x, v) = F0`, and use it with
`simulate`. How many lines of existing code did you have to change? Now do the same for the `force_of` version with
the `if/elif` chain.

</div>

In [ ]:
# your code here

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Interfaces you already use</b></p>

1. Find one interface (a method or a protocol) that several classes of `numpy`,
   `matplotlib` or the standard library share, and name three of them.
2. Argue **against** using one object per event for a Monte Carlo generator that
   produces 10 million events. What would you use instead?

</div>

<details>
<summary><b>Solution</b></summary>

1. For example, anything with `len()` and indexing (lists, tuples, arrays, strings), or
   matplotlib's `Axes.plot`, which accepts lists, tuples, arrays and pandas Series alike.
2. Ten million Python objects cost memory and time (section *The cost, in physics*).
   Better: one object, `EventSample`, holding numpy arrays of energies, positions…, with
   methods that act on all events at once; or a generator that yields blocks of events.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Class or function?</b></p>

Two classes from a real analysis code. What would you do with each one, and why?

```python
class Calibration:
    def __init__(self, slope, offset):
        self.slope, self.offset = slope, offset

    def apply(self, adc):
        return self.slope * adc + self.offset

class Peak:
    def __init__(self, mean, sigma, area):
        self.mean = mean
        self.sigma = sigma
        self.area = area
```

</div>

<details>
<summary><b>Solution</b></summary>

`Calibration` is a borderline case: two parameters and one method. A function with two
parameters (or `functools.partial`) does the same; a frozen dataclass is better if the
calibration is stored, printed and compared. The class is justified only if it gains more
methods (the inverse, the error). `Peak` is data with names and no behaviour: a
`@dataclass` gives the same in a few lines, plus `__repr__` and `__eq__` for free (and `frozen=True` if a fitted peak
should not change).

</details>

## 13. Summary

- Classes are not about elegance; they solve three concrete problems.
- **Binding data together** removes a whole family of silent bugs.
- **Invariants** get one place to live, instead of being everyone's responsibility.
- **Polymorphism is the real payoff**: code that uses an object need not know what it is.
- You are already using objects constantly, whether or not you write any.
- A class with no state and one method should be a function.
- In numerical work: objects for the structure, **arrays for the numbers**.
- Rewriting code into classes must not change its results: test before you refactor.

<details>
<summary><b>[i] Cheat sheet — When to write a class</b></summary>

| Tool | Use it when |
|:--|:--|
| a **function** | no state to keep |
| a **tuple** or **dict** | throwaway grouping, used in one place |
| `@dataclass` / `NamedTuple` | data with names, little or no behaviour |
| a **class** | state *and* behaviour, invariants, several implementations of one interface |
| **numpy arrays** inside a few objects | many identical things |

**Write a class** if at least one is *yes*: several pieces of data always travel together;
something must always be true about them; there will be more than one implementation of
the same idea; the thing has a lifetime (created, used, changed, closed).

</details>

### Check yourself

1. Name the three problems that classes solve.
2. What is polymorphism, in one sentence, and why does it matter for a simulation?
3. When is a class the wrong answer?
4. Why is one object per particle a bad design for ten million particles?
5. What does "prefer composition to inheritance" mean?

<details>
<summary><b>Solutions</b></summary>

1. Data that belongs together travels apart; invariants with no place to live; the same
   operation on different things (`if/elif` chains).
2. Different objects with the same interface: the code that uses them does not need to
   know which one it has, so new models need no change in `simulate`.
3. When there is no state to keep, no invariant and no second implementation: a class
   with one method and no data is a function.
4. Memory overhead per object and a Python loop; use a few objects that hold arrays.
5. Build objects that *have* parts, rather than deep hierarchies of classes that *are*
   kinds of each other.

</details>

Next: the syntax, in [Using classes](./classes_intro.ipynb).